# 05. A01/B01/C01 Baseline 실행

현재 Core의 Pose와 원판 모델을 A/B/C pilot 영상에 같은 설정으로 실행합니다. 모델은 각각 한 번만 불러오고, 영상은 한 장씩 처리합니다. 이 노트북은 후보 생성 → Pose 자동 선택 → 초기 원판 후보 확인 → 원판 track 생성 순서로 실행합니다.

In [ ]:
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import json
import sys

import cv2
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from snatch_analysis.config import ARTIFACT_ROOT, B00_VIDEO_ID, POSE_MODEL_NAME
from snatch_analysis.pipeline import save_frame_analysis, save_json_atomically
from snatch_analysis.plate import (
    aggregate_plate_evaluations,
    evaluate_plate_video,
    infer_plate_video_candidates,
    save_plate_evaluation_results,
    save_plate_candidate_cache,
    save_plate_track,
    select_primary_plate_track,
    validate_pilot_ground_truth,
)
from snatch_analysis.pose import (
    calculate_joint_angles,
    infer_pose_video_candidates,
    resolve_inference_device,
    save_pose_candidate_cache,
    save_selected_pose_keypoints,
    select_initial_primary_lifter,
    select_primary_lifter_track,
    verify_ultralytics_safe_load,
)
from snatch_analysis.video import iter_video_frames, read_video_frame

verify_ultralytics_safe_load()
from ultralytics import YOLO

## 1. 설정

처음에는 `RUN_U9_CANDIDATE_INFERENCE=True`로 후보를 저장합니다. 세 영상의 초기 원판 후보 그림을 확인한 뒤 `PRIMARY_PLATE_CHOICES`를 채우고 `BUILD_CONFIRMED_PLATE_TRACKS=True`로 바꿉니다. 한 실험에서는 모델·입력 크기·confidence 중 한 값만 바꿉니다.

In [ ]:
RUN_U9_CANDIDATE_INFERENCE = False
BUILD_CONFIRMED_PLATE_TRACKS = True
REQUESTED_DEVICE = "mps"

POSE_IMAGE_SIZE = 640
POSE_CONFIDENCE = 0.25
KEYPOINT_CONFIDENCE_THRESHOLD = 0.50
PLATE_IMAGE_SIZE = 320
PLATE_CONFIDENCE = 0.1

EVALUATION_SPLIT_PATH = PROJECT_ROOT / "data/splits/evaluation_videos.csv"
PLATE_WEIGHTS_PATH = PROJECT_ROOT / ARTIFACT_ROOT / B00_VIDEO_ID / "plate_smoke/weights/best.pt"
BASELINE_ARTIFACT_ROOT = PROJECT_ROOT / ARTIFACT_ROOT / "u9_baseline_conf010"

# tuple은 확정 후보, None은 주동작 원판 미탐지입니다.
# 예: {"a01_sn_f_01": (0, 1), "b01_sn_f_01": None}
PRIMARY_PLATE_CHOICES = {
    "a01_sn_f_01": (3, 0),
    "b01_sn_f_01": (12, 0),
    "c01_sn_f_01": (18, 0),
}

In [ ]:
evaluation_split = pd.read_csv(EVALUATION_SPLIT_PATH)
pilot_rows = (
    evaluation_split.loc[
        evaluation_split["split_role"] == "model_selection_pilot"
    ]
    .sort_values("group")
    .reset_index(drop=True)
)
if len(pilot_rows) != 3 or set(pilot_rows["group"]) != {"A", "B", "C"}:
    raise ValueError("model_selection_pilot은 A/B/C 영상이 각각 하나씩 필요합니다.")
if not set(pilot_rows["visible_body_side"]).issubset({"left", "right"}):
    raise ValueError("pilot visible_body_side를 먼저 left/right로 라벨링하세요.")
if set(pilot_rows["read_status"]) != {"ok"}:
    raise ValueError("읽기 오류가 있는 pilot 영상은 baseline에서 제외해야 합니다.")

display(pilot_rows[["video_id", "group", "frame_count", "fps", "visible_body_side"]])

## 2. Raw pilot frame 확인

모델 실행 전에 A/B/C 원본 frame 0을 확인합니다. 선수, 촬영면, 영상 방향이 맞는지 본 뒤에만 `RUN_U9_CANDIDATE_INFERENCE=True`로 바꾸세요.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(18, 7))
for axis, row in zip(axes, pilot_rows.itertuples(index=False), strict=True):
    frame_index = 0
    frame = read_video_frame(PROJECT_ROOT / str(row.path), frame_index)
    axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axis.set_title(f"{row.group} | {row.video_id} | raw frame {frame_index}")
    axis.axis("off")
figure.tight_layout()
plt.show()

## 3. 후보 추론과 Pose 자동 선택

Pose는 화면에서 가장 크고 가로 중앙에 가까운 후보가 일치할 때만 자동 확정합니다. 일치하지 않으면 임의의 사람을 고르지 않고 실패 사유를 남깁니다. 원판은 모든 후보를 먼저 저장합니다.

In [ ]:
device_info = resolve_inference_device(REQUESTED_DEVICE)
actual_device = device_info["actual_device"]
baseline_rows = []

if RUN_U9_CANDIDATE_INFERENCE:
    if not PLATE_WEIGHTS_PATH.exists():
        raise FileNotFoundError(f"원판 baseline 가중치가 없습니다: {PLATE_WEIGHTS_PATH}")
    plate_weights_sha256 = sha256(PLATE_WEIGHTS_PATH.read_bytes()).hexdigest()
    pose_model = YOLO(POSE_MODEL_NAME)
    plate_model = YOLO(str(PLATE_WEIGHTS_PATH))

    for row in pilot_rows.itertuples(index=False):
        video_id = str(row.video_id)
        video_path = PROJECT_ROOT / str(row.path)
        output_directory = BASELINE_ARTIFACT_ROOT / video_id
        output_directory.mkdir(parents=True, exist_ok=True)
        for stale_name in (
            "pose_candidates.csv", "pose_keypoints.csv", "joint_angles.csv",
            "plate_candidates.csv", "plate_track.csv", "run_summary.json",
        ):
            (output_directory / stale_name).unlink(missing_ok=True)
        summary = {
            "video_id": video_id,
            "group": str(row.group),
            "frame_count": int(row.frame_count),
            "device": device_info,
            "pose": {"status": "not_run"},
            "plate": {"status": "not_run", "track_status": "awaiting_confirmation"},
        }

        try:
            started = perf_counter()
            pose_candidates, pose_frame_count = infer_pose_video_candidates(
                pose_model,
                video_path=video_path,
                video_id=video_id,
                frame_count=int(row.frame_count),
                fps=float(row.fps),
                image_size=POSE_IMAGE_SIZE,
                confidence=POSE_CONFIDENCE,
                device=actual_device,
            )
            save_pose_candidate_cache(
                pose_candidates, output_directory / "pose_candidates.csv"
            )
            primary = select_initial_primary_lifter(
                pose_candidates, frame_width=int(row.width)
            )
            if primary["candidate_id"] is None:
                raise ValueError(
                    "주동작자 자동 선택 실패: " + str(primary["reason"])
                )
            pose_keypoints = select_primary_lifter_track(
                pose_candidates,
                frame_count=int(row.frame_count),
                initial_frame_index=int(primary["frame_index"]),
                initial_candidate_id=int(primary["candidate_id"]),
                keypoint_confidence_threshold=KEYPOINT_CONFIDENCE_THRESHOLD,
                fps=float(row.fps),
            )
            save_selected_pose_keypoints(
                pose_keypoints, output_directory / "pose_keypoints.csv"
            )
            joint_angles = calculate_joint_angles(
                pose_keypoints, body_side=str(row.visible_body_side)
            )
            save_frame_analysis(
                joint_angles, output_directory / "joint_angles.csv"
            )
            summary["pose"] = {
                "status": "complete",
                "model": POSE_MODEL_NAME,
                "image_size": POSE_IMAGE_SIZE,
                "confidence": POSE_CONFIDENCE,
                "processed_frame_count": pose_frame_count,
                "candidate_row_count": len(pose_candidates),
                "initial_selection": primary,
                "elapsed_seconds": perf_counter() - started,
            }
        except Exception as error:
            summary["pose"] = {
                "status": "failed",
                "reason": f"{type(error).__name__}: {error}",
            }

        try:
            started = perf_counter()
            plate_candidates, plate_frame_count = infer_plate_video_candidates(
                plate_model,
                video_path=video_path,
                video_id=video_id,
                frame_count=int(row.frame_count),
                fps=float(row.fps),
                image_width=int(row.width),
                image_height=int(row.height),
                image_size=PLATE_IMAGE_SIZE,
                confidence=PLATE_CONFIDENCE,
                device=actual_device,
            )
            save_plate_candidate_cache(
                plate_candidates, output_directory / "plate_candidates.csv"
            )
            summary["plate"] = {
                "status": "complete",
                "track_status": "awaiting_confirmation",
                "model": str(PLATE_WEIGHTS_PATH.relative_to(PROJECT_ROOT)),
                "weights_sha256": plate_weights_sha256,
                "image_size": PLATE_IMAGE_SIZE,
                "confidence": PLATE_CONFIDENCE,
                "processed_frame_count": plate_frame_count,
                "candidate_row_count": len(plate_candidates),
                "elapsed_seconds": perf_counter() - started,
            }
        except Exception as error:
            summary["plate"] = {
                "status": "failed",
                "track_status": "not_available",
                "reason": f"{type(error).__name__}: {error}",
            }

        save_json_atomically(summary, output_directory / "run_summary.json")
        baseline_rows.append({
            "video_id": video_id,
            "pose_status": summary["pose"]["status"],
            "plate_status": summary["plate"]["status"],
        })
else:
    for row in pilot_rows.itertuples(index=False):
        summary_path = BASELINE_ARTIFACT_ROOT / str(row.video_id) / "run_summary.json"
        if summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            baseline_rows.append({
                "video_id": str(row.video_id),
                "pose_status": summary["pose"]["status"],
                "plate_status": summary["plate"]["status"],
            })
        else:
            baseline_rows.append({
                "video_id": str(row.video_id),
                "pose_status": "not_run",
                "plate_status": "not_run",
            })

display(pd.DataFrame(baseline_rows))

## 4. 초기 원판 후보 확인

각 영상의 첫 탐지 frame에서 들어 올리는 원판의 candidate ID를 확인합니다. 배경 원판을 선택하지 않습니다. 후보가 없으면 좌표를 만들어 채우지 않고 해당 영상의 원판 track을 `N/A`로 둡니다.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(18, 7))
for axis, row in zip(axes, pilot_rows.itertuples(index=False), strict=True):
    video_id = str(row.video_id)
    candidate_path = BASELINE_ARTIFACT_ROOT / video_id / "plate_candidates.csv"
    axis.axis("off")
    if not candidate_path.exists():
        axis.set_title(f"{video_id}: 후보 미생성")
        continue
    candidates = pd.read_csv(candidate_path)
    if candidates.empty:
        axis.set_title(f"{video_id}: 원판 후보 없음")
        continue
    first_frame_index = int(candidates["frame_index"].min())
    first_candidates = candidates.loc[
        candidates["frame_index"] == first_frame_index
    ]
    frame = read_video_frame(PROJECT_ROOT / str(row.path), first_frame_index).copy()
    for candidate in first_candidates.itertuples(index=False):
        start = (int(candidate.bbox_x1), int(candidate.bbox_y1))
        end = (int(candidate.bbox_x2), int(candidate.bbox_y2))
        cv2.rectangle(frame, start, end, (0, 255, 0), 3)
        cv2.putText(
            frame, f"id={int(candidate.candidate_id)}", start,
            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2,
        )
    axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axis.set_title(f"{video_id} | frame {first_frame_index}")
figure.tight_layout()
BASELINE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
review_path = BASELINE_ARTIFACT_ROOT / "initial_plate_candidate_review.png"
figure.savefig(review_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"확인 그림 저장: {review_path}")

In [ ]:
track_rows = []
if BUILD_CONFIRMED_PLATE_TRACKS:
    for row in pilot_rows.itertuples(index=False):
        video_id = str(row.video_id)
        output_directory = BASELINE_ARTIFACT_ROOT / video_id
        summary_path = output_directory / "run_summary.json"
        track_path = output_directory / "plate_track.csv"
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        if summary["plate"].get("status") != "complete":
            track_path.unlink(missing_ok=True)
            summary["plate"].update({
                "track_status": "not_available",
                "track_reason": "plate_inference_not_complete",
            })
            save_json_atomically(summary, summary_path)
            track_rows.append({"video_id": video_id, "track_status": "not_available"})
            continue
        if video_id not in PRIMARY_PLATE_CHOICES:
            track_path.unlink(missing_ok=True)
            summary["plate"]["track_status"] = "awaiting_confirmation"
            save_json_atomically(summary, summary_path)
            track_rows.append({"video_id": video_id, "track_status": "awaiting_confirmation"})
            continue
        choice = PRIMARY_PLATE_CHOICES[video_id]
        if choice is None:
            track_path.unlink(missing_ok=True)
            summary["plate"].update({
                "track_status": "not_available",
                "track_reason": "primary_plate_not_detected",
            })
            save_json_atomically(summary, summary_path)
            track_rows.append({"video_id": video_id, "track_status": "not_available"})
            continue
        try:
            candidates = pd.read_csv(output_directory / "plate_candidates.csv")
            pose_path = output_directory / "pose_keypoints.csv"
            lifter_centers = None
            if summary["pose"].get("status") == "complete" and pose_path.exists():
                pose_keypoints = pd.read_csv(pose_path)
                lifter_centers = (
                    pose_keypoints[["frame_index", "bbox_center_x", "bbox_center_y"]]
                    .dropna()
                    .drop_duplicates("frame_index")
                    .rename(columns={"bbox_center_x": "center_x_px", "bbox_center_y": "center_y_px"})
                )
            initial_frame_index, initial_candidate_id = map(int, choice)
            plate_track = select_primary_plate_track(
                candidates,
                frame_count=int(row.frame_count),
                initial_frame_index=initial_frame_index,
                initial_candidate_id=initial_candidate_id,
                lifter_centers=lifter_centers,
                fps=float(row.fps),
            )
            save_plate_track(plate_track, track_path)
            summary["plate"].update({
                "track_status": "complete",
                "initial_frame_index": initial_frame_index,
                "initial_candidate_id": initial_candidate_id,
                "tracked_frame_count": int((plate_track["status"] == "ok").sum()),
            })
        except Exception as error:
            track_path.unlink(missing_ok=True)
            summary["plate"].update({
                "track_status": "failed",
                "track_reason": f"{type(error).__name__}: {error}",
            })
        save_json_atomically(summary, summary_path)
        track_rows.append({
            "video_id": video_id,
            "track_status": summary["plate"]["track_status"],
        })
else:
    print("초기 후보를 확인한 뒤 PRIMARY_PLATE_CHOICES를 입력하고 BUILD_CONFIRMED_PLATE_TRACKS=True로 바꾸세요.")

if track_rows:
    display(pd.DataFrame(track_rows))

## 5. Confidence threshold 평가

저장된 `0.25`와 `0.10` 결과를 수동 정답 30프레임과 비교합니다. 후보 평가는 detector가 정답 원판을 만들었는지, track 평가는 최종 선택이 정답 원판을 유지했는지를 봅니다. IoU 기준은 결과를 보기 전에 `0.50`으로 고정합니다.

In [ ]:
RUN_U9_THRESHOLD_EVALUATION = True
PLATE_EVALUATION_IOU_THRESHOLD = 0.50
FAILURE_REVIEW_EXPERIMENT = "conf010"
FAILURE_REVIEW_LIMIT = 12

U9_GROUND_TRUTH_PATH = PROJECT_ROOT / "data/annotations/pilot_frame_ground_truth.csv"
THRESHOLD_EVALUATION_ROOT = PROJECT_ROOT / ARTIFACT_ROOT / "u9_threshold_evaluation"
THRESHOLD_EXPERIMENTS = {
    "conf025": {
        "expected_confidence": 0.25,
        "artifact_root": PROJECT_ROOT / ARTIFACT_ROOT / "u9_baseline",
    },
    "conf010": {
        "expected_confidence": 0.10,
        "artifact_root": BASELINE_ARTIFACT_ROOT,
    },
}

In [ ]:
threshold_frame_evaluation = pd.DataFrame()
threshold_comparison = pd.DataFrame()

if RUN_U9_THRESHOLD_EVALUATION:
    u9_ground_truth = validate_pilot_ground_truth(
        pd.read_csv(U9_GROUND_TRUTH_PATH), evaluation_split
    )
    frame_tables = []
    summary_rows = []
    comparison_condition_by_video = {}
    for experiment_id, experiment in THRESHOLD_EXPERIMENTS.items():
        experiment_summaries = []
        for row in pilot_rows.itertuples(index=False):
            video_id = str(row.video_id)
            artifact_directory = Path(experiment["artifact_root"]) / video_id
            candidate_path = artifact_directory / "plate_candidates.csv"
            try:
                candidates = pd.read_csv(candidate_path)
            except FileNotFoundError as error:
                raise FileNotFoundError(f"원판 후보 결과가 없습니다: {candidate_path}") from error
            run_summary_path = artifact_directory / "run_summary.json"
            run_summary = json.loads(run_summary_path.read_text(encoding="utf-8"))
            plate_summary = run_summary["plate"]
            if plate_summary.get("status") != "complete":
                raise ValueError(f"{experiment_id}/{video_id}: plate inference가 완료되지 않았습니다.")
            stored_confidence = float(plate_summary["confidence"])
            expected_confidence = float(experiment["expected_confidence"])
            if abs(stored_confidence - expected_confidence) > 1e-9:
                raise ValueError(f"{experiment_id}/{video_id}: 저장 confidence가 기대값과 다릅니다.")
            weights_sha256 = str(plate_summary.get("weights_sha256", ""))
            try:
                valid_weight_hash = len(weights_sha256) == 64 and int(weights_sha256, 16) >= 0
            except ValueError:
                valid_weight_hash = False
            if not valid_weight_hash:
                raise ValueError(f"{experiment_id}/{video_id}: 유효한 weights_sha256가 필요합니다.")
            comparison_condition = {
                "model": str(plate_summary["model"]),
                "weights_sha256": weights_sha256,
                "image_size": int(plate_summary["image_size"]),
                "device": str(run_summary["device"]["actual_device"]),
                "processed_frame_count": int(plate_summary["processed_frame_count"]),
            }
            previous_condition = comparison_condition_by_video.setdefault(
                video_id, comparison_condition
            )
            if comparison_condition != previous_condition:
                raise ValueError(f"{video_id}: confidence 외 비교 조건이 다릅니다.")
            track_path = artifact_directory / "plate_track.csv"
            track_status = str(plate_summary.get("track_status", ""))
            if track_status == "complete":
                track = pd.read_csv(track_path)
            elif track_status == "not_available":
                track_reason = str(plate_summary.get("track_reason", "")).strip()
                if not track_reason:
                    raise ValueError(f"{experiment_id}/{video_id}: not_available 사유가 필요합니다.")
                if track_path.exists():
                    raise ValueError(f"{experiment_id}/{video_id}: not_available인데 stale track이 남았습니다.")
                track = None
            else:
                raise ValueError(f"{experiment_id}/{video_id}: 원판 track 확정 상태를 확인하세요.")
            video_ground_truth = u9_ground_truth.loc[
                u9_ground_truth["video_id"].astype(str) == video_id
            ]
            frame_evaluation, summary = evaluate_plate_video(
                video_ground_truth,
                candidates,
                track,
                frame_count=int(row.frame_count),
                fps=float(row.fps),
                iou_threshold=PLATE_EVALUATION_IOU_THRESHOLD,
                missing_track_reason=str(plate_summary.get("track_reason") or "no_confirmed_track"),
            )
            frame_evaluation.insert(0, "experiment_id", experiment_id)
            frame_evaluation.insert(1, "confidence_threshold", stored_confidence)
            frame_evaluation.insert(2, "group", str(row.group))
            frame_tables.append(frame_evaluation)

            summary.update({
                "experiment_id": experiment_id,
                "confidence_threshold": stored_confidence,
                "group": str(row.group),
                "video_count": 1,
                "candidate_row_count": len(candidates),
                "model": comparison_condition["model"],
                "weights_sha256": comparison_condition["weights_sha256"],
                "image_size": comparison_condition["image_size"],
                "device": comparison_condition["device"],
                "random_seed": "not_applicable_pretrained_inference",
                "provenance_status": "verified",
                "inference_seconds": float(run_summary.get("plate", {}).get("elapsed_seconds", float("nan"))),
            })
            experiment_summaries.append(summary)

        per_video = pd.DataFrame(experiment_summaries)
        experiment_confidence = float(per_video["confidence_threshold"].drop_duplicates().item())
        overall = aggregate_plate_evaluations(per_video)
        overall.update({
            "experiment_id": experiment_id,
            "confidence_threshold": experiment_confidence,
            "group": "ALL",
            "candidate_row_count": int(per_video["candidate_row_count"].sum()),
            "inference_seconds": float(per_video["inference_seconds"].sum()),
            "model": per_video["model"].drop_duplicates().item(),
            "weights_sha256": per_video["weights_sha256"].drop_duplicates().item(),
            "image_size": int(per_video["image_size"].drop_duplicates().item()),
            "device": per_video["device"].drop_duplicates().item(),
            "random_seed": per_video["random_seed"].drop_duplicates().item(),
            "provenance_status": per_video["provenance_status"].drop_duplicates().item(),
        })
        summary_rows.extend(experiment_summaries)
        summary_rows.append(overall)

    threshold_frame_evaluation = pd.concat(frame_tables, ignore_index=True)
    threshold_comparison = pd.DataFrame(summary_rows)
    save_plate_evaluation_results(
        threshold_frame_evaluation, threshold_comparison, THRESHOLD_EVALUATION_ROOT
    )
    display(
        threshold_comparison[[
            "experiment_id", "group", "video_id", "candidate_recall",
            "track_correct_rate", "wrong_target_rate", "plate_ok_rate",
            "longest_non_ok_gap_frames", "mean_normalized_center_error",
        ]].sort_values(["experiment_id", "group"])
    )
else:
    print("threshold 평가를 실행하려면 RUN_U9_THRESHOLD_EVALUATION=True로 바꾸세요.")

In [ ]:
failure_contact_sheet_path = None
if RUN_U9_THRESHOLD_EVALUATION and not threshold_frame_evaluation.empty:
    failures = threshold_frame_evaluation.loc[
        (threshold_frame_evaluation["experiment_id"] == FAILURE_REVIEW_EXPERIMENT)
        & (threshold_frame_evaluation["plate_present"] == "yes")
        & (threshold_frame_evaluation["track_outcome"] != "correct")
    ].copy()
    failures["review_priority"] = failures["track_outcome"].map({
        "wrong_target": 0, "missing": 1,
    })
    failures = failures.sort_values(
        ["review_priority", "track_iou", "video_id", "frame_index"],
        na_position="last",
    ).head(FAILURE_REVIEW_LIMIT)

    if failures.empty:
        print(f"{FAILURE_REVIEW_EXPERIMENT}: 검토할 track 실패가 없습니다.")
    else:
        metadata_by_video = pilot_rows.set_index("video_id")
        review_frames = {}
        for video_id, video_failures in failures.groupby("video_id"):
            metadata = metadata_by_video.loc[video_id]
            target_indices = set(video_failures["frame_index"].astype(int))
            for frame_index, _, frame in iter_video_frames(
                PROJECT_ROOT / str(metadata["path"]),
                expected_frame_count=int(metadata["frame_count"]),
            ):
                if frame_index in target_indices:
                    review_frames[(video_id, frame_index)] = frame.copy()

        column_count = 3
        row_count = (len(failures) + column_count - 1) // column_count
        figure, axes = plt.subplots(row_count, column_count, figsize=(15, 5 * row_count), squeeze=False)
        for axis in axes.flat:
            axis.axis("off")
        for axis, failure in zip(axes.flat, failures.itertuples(index=False)):
            frame = review_frames[(failure.video_id, int(failure.frame_index))]
            ground_truth_start = (int(failure.ground_truth_bbox_x1), int(failure.ground_truth_bbox_y1))
            ground_truth_end = (int(failure.ground_truth_bbox_x2), int(failure.ground_truth_bbox_y2))
            cv2.rectangle(frame, ground_truth_start, ground_truth_end, (0, 255, 0), 4)
            if failure.track_status == "ok":
                track_start = (int(failure.track_bbox_x1), int(failure.track_bbox_y1))
                track_end = (int(failure.track_bbox_x2), int(failure.track_bbox_y2))
                cv2.rectangle(frame, track_start, track_end, (0, 0, 255), 4)
            axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            axis.set_title(
                f"{failure.video_id} frame={int(failure.frame_index)} {failure.track_outcome}"
            )
        figure.suptitle("Green: ground truth | Red: selected track")
        figure.tight_layout()
        failure_contact_sheet_path = (
            THRESHOLD_EVALUATION_ROOT
            / f"{FAILURE_REVIEW_EXPERIMENT}_failure_contact_sheet.png"
        )
        figure.savefig(failure_contact_sheet_path, dpi=140, bbox_inches="tight")
        plt.show()
        print(f"실패 contact sheet 저장: {failure_contact_sheet_path}")